In [9]:
from pathlib import Path
import sys

def ensure_tests_on_path(max_up=10):
    p = Path.cwd().resolve()
    for _ in range(max_up):
        if (p / 'tests').is_dir():
            sys.path.insert(0, str(p))
            return str(p)
        if p.parent == p:
            break
        p = p.parent
    # fallback: insert cwd
    sys.path.insert(0, str(Path.cwd().resolve()))
    return str(Path.cwd().resolve())

root_added = ensure_tests_on_path()
print("Added to sys.path:", root_added)


Added to sys.path: /Users/ashmi/Code/Scripts/phd/travel-crs


In [10]:
COMPARISON_MAP = {
    "much more l1 than l2": 2,
    "much more l2 than l1": -2,
    "slightly more l1": 1,
    "slightly more l1 than l2": 1,
    "slightly more l2 than l1": -1,
    "slightly more l2": -1,
    "about the same": 0,
    "not sure / don't know": -3,
    "not sure / don’t know": -3,
    "not sure": -3,
    "don't know": -3
}


In [38]:
import json
from tests.analysis.src.quantitative import *
from tests.analysis.src.utils import load_listwise_evaluations_df, add_numeric_scores

In [4]:
%load_ext autoreload
%autoreload 2

In [49]:
def show_stats(version="v1"):

    if version == "v1":
        v_text = "L1: Gemini-2.5-flash, L2: GPT-4o-mini"
    else:
        v_text = "L1: Claude-4-sonnet, L2: Qwen3-80B"
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'
    
    df_gpt = load_listwise_evaluations_df(
        gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    print(f"\n=== Confidence-weighted Mean Scores GPT: {version} ===")
    print(compute_stats_scores(df_gpt, with_confidence=True))
    
    print(f"\n=== Confidence-weighted Mean Scores Gemini {version}===")
    print(compute_stats_scores(df_gemini, with_confidence=True))
    
    print(f"\n=== Confidence-weighted Mean Scores Deepseek {version}===")
    print(compute_stats_scores(df_deepseek, with_confidence=True))

    compute_kruskal_test(df_gpt, df_gemini, df_deepseek)
    print("\n\n============ xxxxxxx ============")


In [50]:
show_stats(version="v1")
show_stats(version="v2")

ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v1.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v1.

=== Confidence-weighted Mean Scores GPT: v1 ===
            dimension      mean  variance  count
0           relevance  0.173472  1.292268    101
1           diversity -0.094447  1.081008    101
2  popularity_balance -0.682428  1.207638    101
3      sustainability -0.295877  1.340076    100

=== Confidence-weighted Mean Scores Gemini v1===
            dimension      mean  variance  count
0           relevance  0.458176  1.135461    101
1           diversity -0.150776  1.381923    101
2  popularity_balance -0.532746  1.529060    101
3      sustainability -0.412670  1.453741    101

=== Confidence-weighted Mean Scores Deepseek v1===
            dimension      mean  variance  count
0           relevance  0.392426  0.612742    101
1           diversity  0.006120  0.676829    101
2 

In [18]:
df_gpt.columns

Index(['query_id', 'query', 'judge_model', 'experiment', 'rec_llm_L1',
       'rec_llm_L2', 'best_list', 'overall_explanation', 'overall_confidence',
       'relevance_comparison', 'relevance_explanation', 'relevance_confidence',
       'diversity_comparison', 'diversity_explanation', 'diversity_confidence',
       'sustainability_comparison', 'sustainability_explanation',
       'sustainability_confidence', 'popularity_balance_comparison',
       'popularity_balance_explanation', 'popularity_balance_confidence',
       'relevance_score', 'diversity_score', 'sustainability_score',
       'popularity_balance_score'],
      dtype='object')

## Agreement

In [32]:
version="v1"
def agreement_analysis(version="v1"):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'

    print("=========\n\n")
    
    df_gpt = load_listwise_evaluations_df(
            gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    compute_agreement(df_gpt, df_gemini, df_deepseek)

In [36]:
agreement_analysis("v1")



ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v1.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v1.

Agreement Analysis:

=== RELEVANCE ===
All three agree (GPT=Gem=DS): 5 / 101 (4.95%)
GPT=Gem ≠ DS:                36 / 101 (35.64%)
GPT=DS ≠ Gem:                2 / 101 (1.98%)
Gem=DS ≠ GPT:                21 / 101 (20.79%)
No agreement:                 37 / 101 (36.63%)
Fleiss' Kappa for dimension 'relevance': -0.1565: Poor agreement

=== DIVERSITY ===
All three agree (GPT=Gem=DS): 21 / 101 (20.79%)
GPT=Gem ≠ DS:                25 / 101 (24.75%)
GPT=DS ≠ Gem:                18 / 101 (17.82%)
Gem=DS ≠ GPT:                11 / 101 (10.89%)
No agreement:                 26 / 101 (25.74%)
Fleiss' Kappa for dimension 'diversity': -0.1010: Poor agreement

=== POPULARITY_BALANCE ===
All three agree (GPT=Gem=DS): 14 / 101 (13.86%)
GPT=Gem ≠ DS:                24 / 101 (23.76%)
GPT=DS

/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_va

In [37]:
agreement_analysis("v2")



ℹ️ Loaded 101 successful evaluation entries for gpt5, version v2.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v2.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v2.

Agreement Analysis:

=== RELEVANCE ===
All three agree (GPT=Gem=DS): 5 / 101 (4.95%)
GPT=Gem ≠ DS:                17 / 101 (16.83%)
GPT=DS ≠ Gem:                5 / 101 (4.95%)
Gem=DS ≠ GPT:                22 / 101 (21.78%)
No agreement:                 52 / 101 (51.49%)
Fleiss' Kappa for dimension 'relevance': -0.1896: Poor agreement

=== DIVERSITY ===
All three agree (GPT=Gem=DS): 10 / 101 (9.90%)
GPT=Gem ≠ DS:                35 / 101 (34.65%)
GPT=DS ≠ Gem:                20 / 101 (19.80%)
Gem=DS ≠ GPT:                9 / 101 (8.91%)
No agreement:                 27 / 101 (26.73%)
Fleiss' Kappa for dimension 'diversity': -0.0958: Poor agreement

=== POPULARITY_BALANCE ===
All three agree (GPT=Gem=DS): 16 / 101 (15.84%)
GPT=Gem ≠ DS:                20 / 101 (19.80%)
GPT=DS ≠ 

/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_valid.dimension == dim]["numeric_score"].dropna()
/Users/ashmi/Code/Scripts/phd/travel-crs/tests/analysis/src/quantitative.py:128: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  scores = df_valid[df_va

## Pairwise Correlation

In [46]:
version="v1"
def show_correlations(version="v1"):
    gemini_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gemini_eval_{version}.json'
    gpt_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/gpt5_eval_{version}.json'
    deepseek_eval_path = f'../../data/conv-trs/ecir-2026/direct-reasoner/deepseek_eval_{version}.json'
    
    print("=========\n\n")
    
    df_gpt = load_listwise_evaluations_df(
            gpt_eval_path, model_name="gpt5", version=version)
    df_gemini = load_listwise_evaluations_df(
        gemini_eval_path, model_name="gemini2.5pro", version=version)
    df_deepseek = load_listwise_evaluations_df(deepseek_eval_path, model_name="deepseek", version=version)
    
    print("\n Models: GPT, Gemini")
    print(pairwise_correlation(df_gpt, df_gemini, "GPT", "Gemini"))
    
    print("\n Models: GPT, Deepseek")
    print(pairwise_correlation(df_gpt, df_deepseek, "GPT", "Deepseek"))
    
    print("\n Models: Gemini, Deepseek")
    print(pairwise_correlation(df_gemini, df_deepseek, "Gemini", "Deepseek"))


In [47]:
show_correlations("v1")



ℹ️ Loaded 101 successful evaluation entries for gpt5, version v1.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v1.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v1.

 Models: GPT, Gemini
            dimension  pearson_r       p_value significance
0           relevance   0.665416  3.184653e-14  Significant
1           diversity   0.768077  7.087158e-21  Significant
2  popularity_balance   0.437422  4.786049e-06  Significant
3      sustainability   0.684532  4.086384e-15  Significant

 Models: GPT, Deepseek
            dimension  pearson_r       p_value     significance
0           relevance   0.177517  7.574081e-02  Not significant
1           diversity   0.183334  6.648568e-02  Not significant
2  popularity_balance   0.403583  2.852713e-05      Significant
3      sustainability   0.509173  1.600924e-07      Significant

 Models: Gemini, Deepseek
            dimension  pearson_r       p_value     significance
0           relevance   0.33572

In [48]:
show_correlations("v2")



ℹ️ Loaded 101 successful evaluation entries for gpt5, version v2.
ℹ️ Loaded 101 successful evaluation entries for gemini2.5pro, version v2.
ℹ️ Loaded 101 successful evaluation entries for deepseek, version v2.

 Models: GPT, Gemini
            dimension  pearson_r       p_value significance
0           relevance   0.510967  4.770541e-08  Significant
1           diversity   0.740172  1.383592e-18  Significant
2  popularity_balance   0.456225  1.628702e-06  Significant
3      sustainability   0.727520  1.480811e-17  Significant

 Models: GPT, Deepseek
            dimension  pearson_r       p_value     significance
0           relevance   0.176239  7.790684e-02  Not significant
1           diversity   0.380297  8.751879e-05      Significant
2  popularity_balance   0.601026  3.029194e-11      Significant
3      sustainability   0.464887  1.608669e-06      Significant

 Models: Gemini, Deepseek
            dimension  pearson_r       p_value significance
0           relevance   0.307426  1

## Self Bias